<a href="https://colab.research.google.com/github/Rithikaa7/PrimeLine-Hope_Speech_Detection/blob/main/Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Set up the environment
We start by preparing the Colab environment for training.


### Step 2: Upload datasets
We upload the fine-grained hope speech datasets.


### Step 3: Install libraries
We install the libraries needed for model training.


In [ ]:
!pip install transformers datasets accelerate scikit-learn


### Step 4: Import libraries
We import all necessary Python packages.


In [ ]:
import pandas as pd

train_df = pd.read_csv("train_CG.csv")
dev_df = pd.read_csv("dev_CG.csv")
test_df = pd.read_csv("test_data_withoutlabelCG.csv")

print(train_df.head())


                                                Text              Label
0      Santhu nu masth miss malthondu ullar bro 🤔🤔🤔🤔         uninvolved
1   Undhe n YouTube d thooyere andala mobile bodathe  discouraging hope
2  ತಟ್ಟಿ ನುಂಗೆಲ್ ತಾರೆದ ಮಡಲ್‌ನ್ ನೀರ್‌ಡ್ ಪಾಡ್ದ್ ಅವು...         uninvolved
3                            Super comedy deepak rai   encouraging hope
4                Namma tulundadu comedy first eppodu         uninvolved


### Step 5: Initialize model and tokenizer
Here, we load a pretrained multilingual transformer model (XLM-RoBERTa) and prepare the tokenizer.  
We also map the hope tone labels to numerical IDs so the model can learn them.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

labels = train_df["Label"].unique().tolist()
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label
)

print(label2id)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'uninvolved': 0, 'discouraging hope': 1, 'encouraging hope': 2, 'blended hope': 3}


### Step 6: Tokenize the text data
In this step, we convert the text comments into tokens that the model understands.  
Padding and truncation are applied to handle varying text lengths.


In [ ]:
def tokenize_data(texts):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=128
    )

train_encodings = tokenize_data(train_df["Text"])
dev_encodings = tokenize_data(dev_df["Text"])


### Step 7: Prepare training and validation labels
We convert the textual hope tone labels into numeric form using the label mapping created earlier.


In [ ]:
import torch

train_labels = torch.tensor([label2id[l] for l in train_df["Label"]])
dev_labels = torch.tensor([label2id[l] for l in dev_df["Label"]])


### Step 8 Import PyTorch
We import PyTorch to support dataset creation and model training.


In [ ]:
import torch

### Step 9: Create custom dataset class
We define a dataset class that combines tokenized text and labels,  
allowing the Trainer to efficiently load data during training.


In [ ]:
class HopeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

train_dataset = HopeDataset(train_encodings, train_labels)
dev_dataset = HopeDataset(dev_encodings, dev_labels)


### Step 10: Train the model
We fine-tune the model on the training data using the HuggingFace Trainer.  
The model is trained for a small number of epochs to prevent overfitting.


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_strategy="no",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset
)

trainer.train()


Step,Training Loss
50,1.241515
100,1.168257
150,1.203199
200,1.114348
250,1.167105
300,1.127841
350,1.169052
400,1.160771
450,1.129510
500,1.112889


Step,Training Loss
50,1.241515
100,1.168257
150,1.203199
200,1.114348
250,1.167105
300,1.127841
350,1.169052
400,1.160771
450,1.129510
500,1.112889


TrainOutput(global_step=1498, training_loss=1.1052264428743532, metrics={'train_runtime': 338.448, 'train_samples_per_second': 35.403, 'train_steps_per_second': 4.426, 'total_flos': 788163319277568.0, 'train_loss': 1.1052264428743532, 'epoch': 2.0})

### Step 11: Prepare test data
Now we convert the test comments into a format that the model understands.
No labels here — only text.


In [ ]:
test_encodings = tokenizer(
    test_df["Text"].tolist(),
    padding=True,
    truncation=True,
    max_length=128
)


### Step 12: Create test dataset
We wrap the test data so the trained model can read it properly.


In [ ]:
test_dataset = HopeDataset(test_encodings)


### Step 13: Predict labels
The model now predicts the hope tone for each test comment.


In [ ]:
import numpy as np
predictions = trainer.predict(test_dataset)
pred_ids = np.argmax(predictions.predictions, axis=1)
pred_labels = [id2label[i] for i in pred_ids]


### Step 14: Create submission file
We prepare the final CSV file in the exact format required for submission.


In [ ]:
submission = pd.DataFrame({
    "id": range(1, len(pred_labels) + 1),
    "label": pred_labels
})

submission.head()


,id,label
0,1,encouraging hope
1,2,uninvolved
2,3,encouraging hope
3,4,uninvolved
4,5,encouraging hope


### Step 15: Save CSV
This file will be uploaded for evaluation.


In [ ]:
submission.to_csv("PrimeLine_Tulu_task1.csv", index=False)
